In [ ]:
!pip install --upgrade pip
!pip install pandas==3.0.2

In [ ]:
import pandas as pd
import numpy as np
import time

print(f"Pandas version: {pd.__version__}")

# Создадим датасет на 5 млн строк для тестов
np.random.seed(42)
N = 5_000_000
df = pd.DataFrame({
    'id': np.arange(N),
    'value': np.random.randn(N),
    'category': np.random.choice(['A', 'B', 'C', 'D'], N),
    'text': np.random.choice(['foo', 'bar', 'baz', 'qux'], N)
})
print(f"Создано строк: {len(df):,}")
print(f"Занимаемая память: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Pandas version: 3.0.2
Создано строк: 5,000,000
Занимаемая память: 171.7 MB


## apply vs векторизация
apply с лямбдой медленный, потому что работает построчно через Python.
Векторизованные операции выполняются на уровне C (это такой низкоуровневый язык программирования).

In [ ]:
# apply — медленно
start = time.time()
df['double_apply'] = df['value'].apply(lambda x: x * 2)
print(f"apply: {time.time() - start:.2f} сек")

# векторизация — быстро
start = time.time()
df['double_vec'] = df['value'] * 2
print(f"Векторизация: {time.time() - start:.2f} сек")

apply: 2.56 сек
Векторизация: 0.02 сек


## Чтение CSV vs Parquet
Parquet хранит типы и сжимает данные. Проверим на 1 млн строк.

In [ ]:
N = 1_000_000
test = pd.DataFrame({'a': np.random.randn(N), 'b': np.random.randn(N)})
test.to_csv('/tmp/test.csv', index=False)
test.to_parquet('/tmp/test.parquet')

# CSV
start = time.time()
pd.read_csv('/tmp/test.csv')
print(f"CSV: {time.time() - start:.2f} сек")

# Parquet
start = time.time()
pd.read_parquet('/tmp/test.parquet')
print(f"Parquet: {time.time() - start:.2f} сек")

CSV: 0.55 сек
Parquet: 0.21 сек


## Pandas однопоточный
Запусти groupby и посмотри в диспетчере задач — загружено только одно ядро.

In [ ]:
start = time.time()
df.groupby('category')['value'].mean()
print(f"groupby: {time.time() - start:.2f} сек")

groupby: 0.32 сек


## Чтение большого CSV по частям
Если файл не влезает в память, используем chunksize.

In [ ]:
# Создадим CSV побольше
test.to_csv('/tmp/large.csv', index=False)

total = 0
for chunk in pd.read_csv('/tmp/large.csv', chunksize=100000):
    total += len(chunk)
print(f"Всего строк прочитано: {total:,}")

Всего строк прочитано: 1,000,000


## Экономия памяти правильными типами
По умолчанию Pandas использует float64 (8 байт) для чисел с плавающей точкой.
Мы можем сократить вдвое, перейдя на float32.

In [ ]:
df['value32'] = df['value'].astype('float32')
print(f"float64: {df['value'].memory_usage(deep=True) / 1024**2:.1f} MB")
print(f"float32: {df['value32'].memory_usage(deep=True) / 1024**2:.1f} MB")

float64: 38.1 MB
float32: 19.1 MB




---

